# FireSight Phase 3b - detector learning-curve pilot (Kaggle, T4)

**Purpose:** does more training data still buy recall at a fixed false-alarm rate?

**Base weights:** `yolo11s.pt` for all three runs (downloaded at runtime). Licence: **AGPL-3.0** (Ultralytics default) - commercial distribution may require an Ultralytics Enterprise Licence; record before shipping.

**Rules**
- Exactly **one GPU notebook at a time** (free quota ~30 h/week). The preflight cell refuses to continue if more than one GPU is visible.
- Selection data is `data/splits/val.jsonl` slices only. **bench_R, TEST and testbench100 are not in this dataset** (see `LEAKAGE_REPORT.json`).
- Every epoch is checkpointed to `/kaggle/working`; after a session restart re-run the train cell and it resumes from `last.pt`.
- Do **not** upload anything from here. The dataset is private.

**Estimated GPU hours (yolo11s @640, batch 64, AMP, T4)** - to be replaced by measured `images/s`:

| run | images | epochs | est. @60 img/s | est. @100 img/s |
|---|---:|---:|---:|---:|
| P10 | 5,898 | 15 | 0.5 h | 0.3 h |
| P25 | 14,747 | 15 | 1.1 h | 0.7 h |
| P50 | 29,493 | 15 | 2.2 h | 1.4 h |
| **pilot total** | | | **~3.8 h** | **~2.4 h** |

**Credentials:** never paste a Kaggle token into a notebook or the repo. `kaggle.json` / `KAGGLE_KEY` are read by the CLI only, from outside the repo.


In [ ]:
import json, os, subprocess, sys
import torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 x1 in the notebook settings"
n = torch.cuda.device_count()
print("torch", torch.__version__, "cuda", torch.version.cuda, "gpus", n)
for i in range(n):
    print("gpu", i, torch.cuda.get_device_name(i))
assert n >= 1, "No GPU visible"
# Kaggle may expose 2 T4s in one session; training is pinned to device 0.
print(f"OK: {n} GPU(s) visible, training pinned to device 0. Run only ONE GPU notebook at a time.")


In [ ]:
# Pinned versions. Ultralytics is NOT in the Kaggle base image; install it and its
# light deps with --no-deps (torch/opencv/numpy already match). Disable ultralytics'
# import-time auto-installer (YOLO_AUTOINSTALL) which can stall on Kaggle.
import importlib.metadata as md, json, os

os.environ["YOLO_AUTOINSTALL"] = "False"
os.environ["YOLO_VERBOSE"] = "False"
os.environ["YOLO_OFFLINE"] = "True"  # no network in the training path

def ver(name):
    try:
        return md.version(name)
    except Exception:
        return None

print("before:", {n: ver(n) for n in ("ultralytics", "onnx", "torch", "numpy", "opencv-python",
                                      "ultralytics-thop", "polars", "py-cpuinfo")}, flush=True)
if ver("ultralytics") != "8.4.147":
    !pip install -q --no-deps --progress-bar off ultralytics==8.4.147 ultralytics-thop polars py-cpuinfo
print("after:", {n: ver(n) for n in ("ultralytics", "ultralytics-thop", "polars", "py-cpuinfo")}, flush=True)
print(json.dumps({"ultralytics": ver("ultralytics"), "onnx": ver("onnx"),
                  "torch": ver("torch"), "numpy": ver("numpy"),
                  "opencv": ver("opencv-python")}, indent=2), flush=True)
# ultralytics is imported inside the training/eval subprocesses so a failure is visible
# in their stdout instead of silently stalling a notebook cell.


In [ ]:
import os, json, sys
from pathlib import Path

# Find the mounted dataset root (Kaggle mounts it under /kaggle/input/<slug>).
def find_dataset_root():
    base = Path("/kaggle/input")
    if not base.exists():
        return None
    for yaml_path in sorted(base.rglob("data.yaml")):
        root = yaml_path.parent
        if (root / "lists").is_dir() and (root / "images").is_dir() and (root / "labels").is_dir():
            return root
    return None

root = find_dataset_root()
if root is None:
    listing = []
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").iterdir():
            listing.append(str(p) + ("/" + ",".join(sorted(x.name for x in p.iterdir())[:8]) if p.is_dir() else ""))
    raise FileNotFoundError(f"dataset not mounted. /kaggle/input = {listing}")

DATASET = str(root)
WORK = "/kaggle/working"

# Code ships as a separate small dataset so fixes do not need a 3 GB re-upload.
code_hits = sorted(Path("/kaggle/input").rglob("pilot_train.py"))
assert code_hits, "pilot_train.py not found under /kaggle/input"
preferred = [p for p in code_hits if "firesight-pilot-code" in str(p)] or code_hits
CODE = str(preferred[0].parent)
assert "def resolve_list" in preferred[0].read_text(), "stale pilot_train.py shipped"
sys.path.insert(0, CODE)
print("CODE =", CODE)
print("DATASET =", DATASET)
print("top-level:", sorted(os.listdir(DATASET)))
print(open(f"{DATASET}/manifest_summary.json").read())
print(open(f"{DATASET}/LEAKAGE_REPORT.json").read())
assert json.load(open(f"{DATASET}/LEAKAGE_REPORT.json"))["verdict"] == "PASS", "Leakage check failed - stop"
for f in ("train_10.txt", "train_25.txt", "train_50.txt", "val_early.txt", "val_cal.txt", "val_report.txt"):
    print(f, sum(1 for l in open(f"{DATASET}/lists/{f}") if l.strip()))
print("images:", len(os.listdir(f"{DATASET}/images")), "labels:", len(os.listdir(f"{DATASET}/labels")))


## 1. Train the three learning-curve points

Each run checkpoints every epoch. If a session dies, re-run this cell: it detects
`last.pt` and passes `--resume`. `P10n` (yolo11n) is an optional pipeline sanity arm.


In [ ]:
# 150 s import probe: if anything hangs we see the last marker and the cell fails fast
# instead of burning the GPU session.
import subprocess

probe = r"""
import os, time
os.environ["YOLO_OFFLINE"] = "True"
os.environ["YOLO_AUTOINSTALL"] = "False"
t = time.time(); print("mark torch", flush=True); import torch; print("torch ok", round(time.time()-t, 1), flush=True)
t = time.time(); print("mark torchvision", flush=True); import torchvision; print("torchvision ok", round(time.time()-t, 1), flush=True)
t = time.time(); print("mark ultralytics", flush=True); import ultralytics; print("ultralytics ok", round(time.time()-t, 1), flush=True)
t = time.time(); print("mark model", flush=True)
from ultralytics import YOLO
m = YOLO("/kaggle/input/firesight-pilot-code/yolo11s.pt")
print("model ok", round(time.time()-t, 1), flush=True)
"""
r = subprocess.run(["timeout", "150", "python", "-u", "-c", probe], capture_output=True, text=True)
print(r.stdout, flush=True)
if r.stderr:
    print(r.stderr[-2000:], flush=True)
assert r.returncode == 0, f"import probe failed or timed out (rc={r.returncode})"


In [ ]:
import os, subprocess
FRACTIONS = ["10", "25", "50"]
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
for frac in FRACTIONS:
    out = f"{WORK}/runs/P{frac}"
    cmd = ["timeout", "10800", "python", "-u", f"{CODE}/pilot_train.py", "--dataset", DATASET,
           "--fraction", frac, "--model", f"{CODE}/yolo11s.pt", "--epochs", "15", "--imgsz", "640",
           "--batch", "64", "--device", "0", "--workers", "2", "--amp", "--out", out]
    if os.path.exists(f"{out}/weights/last.pt"):
        cmd.append("--resume")
    print("RUN:", " ".join(cmd), flush=True)
    # stderr -> stdout: Kaggle's log forwarder drains stdout only; a full stderr pipe
    # would block the child (observed). A hard timeout prevents a 12 h session burn.
    subprocess.run(cmd, check=True, stderr=subprocess.STDOUT, env=env)


## 2. Score each run on the val.jsonl report slice

Thresholds are calibrated on `val_cal` (val.jsonl) for **FPR <= 2% per negative image**,
then scored on the disjoint `val_report` slice with Wilson 95% CIs.


In [ ]:
import gc, subprocess
import torch
from ultralytics import YOLO
from pathlib import Path

def read_list(name):
    # lists are relative to the dataset root (images/<file>) - resolve before predicting
    out = []
    for line in open(f"{DATASET}/lists/{name}"):
        line = line.strip()
        if not line:
            continue
        p = Path(line)
        out.append(p if p.is_absolute() else Path(DATASET) / p)
    return out

def predict_val(weights, imgsz=640):
    paths = read_list("val_cal.txt") + read_list("val_report.txt")
    model = YOLO(weights)
    rows = []
    # batch/half explicit: YOLO(best.pt) inherits training batch=64 otherwise -> CUDA OOM
    for res in model.predict(paths, imgsz=imgsz, conf=0.01, batch=8, half=True,
                             verbose=False, stream=True):
        dets = []
        for b in res.boxes:
            cls = int(b.cls.item())
            dets.append({"label": "fire" if cls == 0 else "smoke",
                         "confidence": round(float(b.conf.item()), 6),
                         "box": [float(v) for v in b.xyxy[0].tolist()]})
        rows.append({"id": Path(res.path).stem, "detections": dets})
    out = f"{WORK}/preds_{Path(weights).parent.parent.name}.jsonl"
    with open(out, "w") as fh:
        for r in rows:
            fh.write(json.dumps(r) + "\n")
    return out

for frac in FRACTIONS:
    weights = f"{WORK}/runs/P{frac}/weights/best.pt"
    if not os.path.exists(weights):
        print("missing", weights); continue
    preds = predict_val(weights)
    del model
    gc.collect(); torch.cuda.empty_cache()
    subprocess.run(["python", f"{CODE}/pilot_eval.py", "--dataset", DATASET, "--preds", preds,
                    "--model", f"P{frac}", "--target-fpr", "0.02",
                    "--out", f"{WORK}/metrics_P{frac}.json"], check=True)


## 3. Summary table and learning curve

In [ ]:
import glob, json
import matplotlib.pyplot as plt

rows = []
for frac in FRACTIONS:
    rp, mp = f"{WORK}/runs/P{frac}/run_summary.json", f"{WORK}/metrics_P{frac}.json"
    if not (os.path.exists(rp) and os.path.exists(mp)):
        continue
    run, met = json.load(open(rp)), json.load(open(mp))
    c = met["splits"]["val"]["combined"]
    rows.append({"fraction": frac, "images": run["n_train"],
                 "images_per_s": run.get("last_epoch_images_per_s") or run["throughput"].get("images_per_s"),
                 "s_per_epoch": run.get("last_epoch_time_s"),
                 "recall": c["recall"], "ci": c["recall_95ci"],
                 "fpr_neg": met["splits"]["val"]["fpr_per_negative_image"]["fpr"],
                 "map50": met["splits"]["val"]["box_ap"]["fire"]["ap50"]})
print(f"{'run':6s} {'images':>7s} {'img/s':>7s} {'s/epoch':>8s} {'recall@FPR2%':>14s} {'95% CI':>18s} {'FPR/neg':>8s}")
for r in rows:
    ci = f"[{r['ci'][0]:.3f}, {r['ci'][1]:.3f}]" if r['ci'] else "-"
    print(f"P{r['fraction']:5s} {r['images']:7d} {str(r['images_per_s']):>7s} {str(r['s_per_epoch']):>8s} "
          f"{r['recall'] if r['recall'] is not None else float('nan'):14.4f} {ci:>18s} {str(r['fpr_neg']):>8s}")
json.dump(rows, open(f"{WORK}/pilot_summary.json", "w"), indent=2)

if rows:
    xs = [r["images"] for r in rows]; ys = [r["recall"] or 0 for r in rows]
    lo = [r["ci"][0] if r["ci"] else 0 for r in rows]; hi = [r["ci"][1] if r["ci"] else 0 for r in rows]
    plt.figure(figsize=(6, 4))
    plt.errorbar(xs, ys, yerr=[[y - l for y, l in zip(ys, lo)], [h - y for y, h in zip(ys, hi)]], marker="o")
    plt.xlabel("training images"); plt.ylabel("val recall @ FPR<=2% / negative")
    plt.title("FireSight pilot learning curve (val.jsonl)")
    plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(f"{WORK}/learning_curve.png", dpi=160); plt.show()


In [ ]:
# ONNX export of the 50% run (deployment format; 10 MB for yolo11n, ~40 MB for yolo11s)
from ultralytics import YOLO
best = f"{WORK}/runs/P50/weights/best.pt"
if os.path.exists(best):
    print(YOLO(best).export(format="onnx", imgsz=640, opset=17, dynamic=False, simplify=False))
else:
    print("P50 best.pt missing; run the training cell first")


## 4. Decision rules (fill in, do not upload)

| outcome | rule | action |
|---|---|---|
| **GO** | recall rises from P10 to P50 by >= 5 pp with non-overlapping CIs, P50 beats the baseline at equal FPR, **and one bench_R DEV-only run per checkpoint (300 images) reproduces the ordering; SEALED untouched** | approve the full run; recompute full-run hours from measured `img/s` |
| **TRIM** | curve rises but projected full run exceeds ~20 GPU h or quota | fewer epochs / 25-50% data / yolo11n, then re-approve |
| **STOP** | P10 -> P50 flat within CI, or no run beats baseline at equal FPR | stop; diagnose label noise / task mismatch / augmentation before more training |

Report: `pilot_summary.json`, `learning_curve.png`, per-run `run_summary.json` and `metrics_P*.json`.
**Nothing is uploaded from this notebook.** Stop here and share the summary.
